# Gemma 3 4B 모델을 LiteRT (TFLite)로 변환하기

이 노트북은 Google의 Gemma 3 4B 모델(텍스트 전용 + 멀티모달)을 LiteRT Edge Generative API를 사용하여
TFLite 모델로 변환하고 모바일 디바이스에서 실행 가능한 형태로 포팅하는 전체 과정을 보여줍니다.

## Gemma 3 4B 모델 특징

| 항목 | 값 |
|------|-----|
| 파라미터 수 | ~4B |
| 레이어 수 | 34 |
| 임베딩 차원 | 2560 |
| 어텐션 헤드 | 8 (head_dim=256) |
| GQA 그룹 (kv_heads) | 4 |
| FFN 크기 | 10240 (Gated, GELU_TANH) |
| Vocab 크기 | 262,208 |
| 최대 시퀀스 | 32,768 |
| 어텐션 패턴 | Global (매 6번째) + Local Sliding Window (1024) |
| Vision Encoder | SigLIP (27 layers, dim=1152) |
| Projection | [1152, 2560] (transpose 필요) |
| mm_norm | RMS_NORM (bias 없음) |

## 워크플로우

1. 환경 설정
2. 체크포인트 다운로드
3. 모델 아키텍처 이해
4. 4B 텍스트 전용 모델 빌드 및 검증
5. 4B 멀티모달 모델 빌드 및 검증
6. TFLite 변환 (텍스트 전용)
7. TFLite 변환 (멀티모달)
8. TFLite 추론 (텍스트 생성)

## 1. 환경 설정

In [ ]:
!pip install litert-torch
!pip install sentencepiece

In [ ]:
import os
import pathlib

import litert_torch
from litert_torch.generative.examples.gemma3 import decoder
from litert_torch.generative.examples.gemma3 import gemma3
from litert_torch.generative.examples.gemma3 import image_encoder
from litert_torch.generative.layers import kv_cache as kv_utils
import litert_torch.generative.layers.attention_utils as attn_utils
import litert_torch.generative.layers.model_config as cfg
from litert_torch.generative.utilities import export_config as export_cfg
from litert_torch.generative.utilities import converter as conv_utils
import litert_torch.generative.utilities.loader as loading_utils
import torch
from torch import nn

print(f"LiteRT Torch version: {litert_torch.__version__}")
print(f"PyTorch version: {torch.__version__}")

## 2. 체크포인트 다운로드

Gemma 3 4B 체크포인트는 HuggingFace에서 다운로드합니다.

```bash
huggingface-cli download google/gemma-3-4b-it --local-dir ~/gemma-3-4b-it/
```

4B 모델은 `language_model.model.layers.{}.xxx` prefix를 사용하는 HuggingFace 형식의 체크포인트를 사용합니다.
멀티모달의 경우 Vision Encoder(`vision_tower.xxx`) 및 Projection(`multi_modal_projector.xxx`) 테서도 함께 포함되어 있습니다.

In [ ]:
# 체크포인트 경로 설정
CHECKPOINT_PATH = os.path.join(pathlib.Path.home(), "gemma-3-4b-it")

OUTPUT_DIR = "/tmp/gemma3_4b_tflite"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Checkpoint exists: {os.path.exists(CHECKPOINT_PATH)}")
print(f"Output dir: {OUTPUT_DIR}")

## 3. Gemma 3 4B 아키텍처 이해

### Decoder (Text)

270M/1B와 동일한 Decoder Block 구조이지만 규모가 다릅니다:
- 34 layers, embedding_dim=2560, num_heads=8, kv_heads=4, ffn=10240
- Sliding window: 1024 (270M/1B는 512)
- vocab_size: 262,208 (config.json은 262,144이지만 실제 embed_tokens shape 기준)

### Multimodal 구성요소

```
Image ─► SigLIP Vision Encoder (27 layers) ─► SigLIP Exit (avg_pool)
      ─► mm_norm (RMS_NORM) ─► Projection [1152→2560] ─► Decoder
```

### 테서 이름 매핑 (HuggingFace 4B)

| 컴포넌트 | Prefix |
|-----------|--------|
| Decoder | `language_model.model.layers.{}.xxx` |
| Vision Encoder | `vision_tower.vision_model.encoder.layers.{}.xxx` |
| Projection | `multi_modal_projector.mm_input_projection_weight` |
| mm_norm | `multi_modal_projector.mm_soft_emb_norm.weight` |

In [ ]:
# 4B Decoder 설정 확인
config_4b = decoder.get_decoder_config_4b()

print("=== Gemma 3 4B Decoder Config ===")
print(f"  vocab_size: {config_4b.vocab_size:,}")
print(f"  num_layers: {config_4b.num_layers}")
print(f"  embedding_dim: {config_4b.embedding_dim}")
print(f"  max_seq_len: {config_4b.max_seq_len:,}")
print(f"  embedding_scale: {config_4b.embedding_scale}")
print(f"  lm_head_use_bias: {config_4b.lm_head_use_bias}")
print()

print("=== Per-Layer Attention Config ===")
for i in range(config_4b.num_layers):
    bc = config_4b.block_config(i)
    ac = bc.attn_config
    attn_type = "GLOBAL" if ac.attn_type == cfg.AttentionType.GLOBAL else "LOCAL"
    print(
        f"  Layer {i:2d}: {attn_type:6s} | "
        f"heads={ac.num_heads}, head_dim={ac.head_dim}, "
        f"kv_heads={ac.num_query_groups}, "
        f"window={ac.sliding_window_size}, "
        f"rotary_base={ac.rotary_base:>10,}"
    )

print()
print("=== FFN Config ===")
ff = config_4b.block_config(0).ff_config
print(f"  type: {ff.type}")
print(f"  intermediate_size: {ff.intermediate_size}")
print(f"  activation: {ff.activation.type}")

In [ ]:
# 4B MM 설정 확인
config_4b_mm = gemma3.get_model_config_4b()

print("=== Gemma 3 4B Multimodal Config ===")
print(f"  image_token_id: {config_4b_mm.image_token_id}")
print(f"  image_projection_scale: {config_4b_mm.image_projection_scale}")
print(f"  image_projection_use_bias: {config_4b_mm.image_projection_use_bias}")
print(f"  mm_norm type: {config_4b_mm.mm_norm_config.type}")
print(f"  mm_extra_tokens: {config_4b_mm.mm_extra_tokens}")
print()

ie_config = config_4b_mm.image_encoder_config
print("=== Image Encoder (SigLIP) Config ===")
print(f"  num_layers: {ie_config.num_layers}")
print(f"  embedding_dim: {ie_config.embedding_dim}")
print(f"  image_size: {ie_config.image_embedding.image_size}")
print(f"  patch_size: {ie_config.image_embedding.patch_size}")
print(f"  num_mm_tokens_per_image: {ie_config.num_mm_tokens_per_image}")
print(f"  attn heads: {ie_config.block_config(0).attn_config.num_heads}")
print(f"  attn head_dim: {ie_config.block_config(0).attn_config.head_dim}")
print(f"  ffn intermediate: {ie_config.block_config(0).ff_config.intermediate_size}")

In [ ]:
# 테서 이름 매핑 확인 (HuggingFace 4B)
print("=== HF 4B Decoder Tensor Names ===")
hf4b = decoder.TENSOR_NAMES_HF_4B
print(f"  embedding:        {hf4b.embedding}")
print(f"  attn_query_proj:  {hf4b.attn_query_proj}")
print(f"  attn_key_proj:    {hf4b.attn_key_proj}")
print(f"  attn_value_proj:  {hf4b.attn_value_proj}")
print(f"  attn_output_proj: {hf4b.attn_output_proj}")
print(f"  attn_query_norm:  {hf4b.attn_query_norm}")
print(f"  attn_key_norm:    {hf4b.attn_key_norm}")
print(f"  ff_gate_proj:     {hf4b.ff_gate_proj}")
print(f"  ff_up_proj:       {hf4b.ff_up_proj}")
print(f"  ff_down_proj:     {hf4b.ff_down_proj}")
print(f"  pre_attn_norm:    {hf4b.pre_attn_norm}")
print(f"  post_attn_norm:   {hf4b.post_attn_norm}")
print(f"  pre_ff_norm:      {hf4b.pre_ff_norm}")
print(f"  post_ff_norm:     {hf4b.post_ff_norm}")
print(f"  final_norm:       {hf4b.final_norm}")
print(f"  lm_head:          {hf4b.lm_head} (shared with embedding)")
print()

print("=== HF 4B Vision Encoder Tensor Names ===")
hf_vis = image_encoder.TENSOR_NAMES_HF
print(f"  embedding:        {hf_vis.embedding}")
print(f"  embedding_pos:    {hf_vis.embedding_position}")
print(f"  attn_query_proj:  {hf_vis.attn_query_proj}")
print(f"  attn_output_proj: {hf_vis.attn_output_proj}")
print(f"  pre_attn_norm:    {hf_vis.pre_attn_norm}")
print(f"  pre_ff_norm:      {hf_vis.pre_ff_norm}")
print(f"  final_norm:       {hf_vis.final_norm}")
print()

print("=== Projection / mm_norm Tensor Names ===")
print(f"  projection: {gemma3.PROJECTION_TENSOR_NAME_4B}")
print(f"  mm_norm:    {gemma3.MM_NORM_TENSOR_NAME_4B}")

## 4. 4B 텍스트 전용 모델 빌드 및 검증

`gemma3.build_model_4b()`는 decoder-only 모델만 빌드합니다:
1. `get_decoder_config_4b()`로 모델 설정 생성
2. `Decoder` 클래스 인스턴스 생성
3. 체크포인트에서 가중치 로드 (HF 4B 테서 이름 매핑 자동 감지)
4. `.eval()` 모드 설정

In [ ]:
# KV 캐시 설정
KV_CACHE_MAX_LEN = 1280
PREFILL_SEQ_LENS = [8, 64, 128, 256, 512, 1024]

print(f"KV cache max length: {KV_CACHE_MAX_LEN}")
print(f"Prefill sequence lengths: {PREFILL_SEQ_LENS}")

In [ ]:
# 4B 텍스트 전용 모델 빌드
if os.path.exists(CHECKPOINT_PATH):
    gemma3_4b_model = gemma3.build_model_4b(
        checkpoint_path=CHECKPOINT_PATH,
        mask_cache_size=KV_CACHE_MAX_LEN,
    )
    total_params = sum(p.numel() for p in gemma3_4b_model.parameters())
    print(f"4B Decoder model loaded successfully!")
    print(f"Total parameters: {total_params:,}")
    print(f"Model size (FP32): ~{total_params * 4 / 1024 / 1024:.0f} MB")
else:
    print(f"Checkpoint not found at: {CHECKPOINT_PATH}")
    print("Please download: huggingface-cli download google/gemma-3-4b-it --local-dir ~/gemma-3-4b-it/")

In [ ]:
# 4B 텍스트 전용 Forward pass 검증

@torch.inference_mode()
def verify_decoder_output(model, config, kv_cache_max_len=1280):
    """Decoder 모델의 forward pass가 정상적으로 동작하는지 확인합니다."""
    test_tokens = torch.tensor(
        [[2, 651, 9456, 576, 573, 3520, 3858, 603, 235248]], dtype=torch.int
    )
    seq_len = test_tokens.shape[1]
    input_pos = torch.arange(0, seq_len, dtype=torch.int)

    kv_cache = kv_utils.KVCache.from_model_config(
        kv_cache_max_len, config,
        kv_layout=kv_utils.KV_LAYOUT_TRANSPOSED,
    )

    mask_cache = attn_utils.build_causal_mask_cache(kv_cache_max_len)
    mask = mask_cache.index_select(2, input_pos)

    output = model(test_tokens, input_pos, kv_cache, mask=mask)

    logits = output["logits"]
    print(f"Output logits shape: {logits.shape}")
    print(f"Expected: (1, {seq_len}, {config.vocab_size})")
    assert logits.shape == (1, seq_len, config.vocab_size), "Shape mismatch!"

    top5 = torch.topk(logits[0, -1], 5)
    print(f"\nTop-5 predicted tokens:")
    for i, (val, idx) in enumerate(zip(top5.values, top5.indices)):
        print(f"  {i+1}. token_id={idx.item():6d}, logit={val.item():.4f}")

    print("\nForward pass verification: PASSED")
    return output


if os.path.exists(CHECKPOINT_PATH):
    verify_decoder_output(gemma3_4b_model, decoder.get_decoder_config_4b(), KV_CACHE_MAX_LEN)
else:
    print("Skipping verification - checkpoint not available.")

## 5. 4B 멀티모달 모델 빌드 및 검증

`gemma3.build_model_4b_mm()`은 다음을 수행합니다:
1. SigLIP Vision Encoder 가중치 로드 (`vision_tower.xxx`)
2. Decoder 가중치 로드 (`language_model.model.xxx`)
3. Projection weight 로드 및 **전치** (`[1152, 2560]` → `[2560, 1152]`)
4. mm_norm (RMS_NORM) weight 로드

In [ ]:
# 4B 멀티모달 모델 빌드
if os.path.exists(CHECKPOINT_PATH):
    gemma3_4b_mm_model = gemma3.build_model_4b_mm(
        checkpoint_path=CHECKPOINT_PATH,
        mask_cache_size=KV_CACHE_MAX_LEN,
    )
    total_params = sum(p.numel() for p in gemma3_4b_mm_model.parameters())
    print(f"4B Multimodal model loaded successfully!")
    print(f"Total parameters: {total_params:,}")
    print(f"Model size (FP32): ~{total_params * 4 / 1024 / 1024:.0f} MB")
    print()

    # 각 컴포넌트별 파라미터 수
    enc_params = sum(p.numel() for p in gemma3_4b_mm_model.image_encoder.parameters())
    dec_params = sum(p.numel() for p in gemma3_4b_mm_model.decoder.parameters())
    proj_params = sum(p.numel() for p in gemma3_4b_mm_model.image_projection.parameters())
    norm_params = sum(p.numel() for p in gemma3_4b_mm_model.mm_norm.parameters())
    print(f"  Image Encoder:  {enc_params:>12,} params")
    print(f"  Decoder:        {dec_params:>12,} params")
    print(f"  Projection:     {proj_params:>12,} params")
    print(f"  mm_norm:        {norm_params:>12,} params")
    print()

    # Projection weight shape 확인
    proj_w = gemma3_4b_mm_model.image_projection.weight
    print(f"  Projection weight shape: {proj_w.shape}")
    print(f"  Expected: ({config_4b_mm.decoder_config.embedding_dim}, {config_4b_mm.image_encoder_config.embedding_dim})")
    assert proj_w.shape == (2560, 1152), f"Projection shape mismatch: {proj_w.shape}"
    print("  Projection shape: OK")
else:
    print(f"Checkpoint not found at: {CHECKPOINT_PATH}")

In [ ]:
# 4B MM 모델 - 텍스트 전용 Forward pass 검증 (pixel_values=None)

@torch.inference_mode()
def verify_mm_text_only(model, config_mm, kv_cache_max_len=1280):
    """MM 모델의 텍스트 전용 forward pass를 검증합니다."""
    test_tokens = torch.tensor(
        [[2, 651, 9456, 576, 573, 3520, 3858, 603, 235248]], dtype=torch.int
    )
    seq_len = test_tokens.shape[1]
    input_pos = torch.arange(0, seq_len, dtype=torch.int)

    dec_config = config_mm.decoder_config
    kv_cache = kv_utils.KVCache.from_model_config(
        kv_cache_max_len, dec_config,
        kv_layout=kv_utils.KV_LAYOUT_TRANSPOSED,
    )

    # pixel_values=None → 텍스트 전용 경로
    output = model(
        tokens=test_tokens,
        input_pos=input_pos,
        kv_cache=kv_cache,
        pixel_values=None,
    )

    logits = output["logits"]
    print(f"MM text-only logits shape: {logits.shape}")
    print(f"Expected: (1, {seq_len}, {dec_config.vocab_size})")
    assert logits.shape == (1, seq_len, dec_config.vocab_size), "Shape mismatch!"

    top5 = torch.topk(logits[0, -1], 5)
    print(f"\nTop-5 predicted tokens:")
    for i, (val, idx) in enumerate(zip(top5.values, top5.indices)):
        print(f"  {i+1}. token_id={idx.item():6d}, logit={val.item():.4f}")

    print("\nMM text-only forward pass: PASSED")
    return output


if os.path.exists(CHECKPOINT_PATH):
    verify_mm_text_only(gemma3_4b_mm_model, config_4b_mm, KV_CACHE_MAX_LEN)
else:
    print("Skipping - checkpoint not available.")

In [ ]:
# 4B MM 모델 - Vision + Text Forward pass 검증

@torch.inference_mode()
def verify_mm_with_image(model, config_mm, kv_cache_max_len=1280):
    """MM 모델의 이미지 포함 forward pass를 검증합니다."""
    ie_config = config_mm.image_encoder_config
    dec_config = config_mm.decoder_config
    num_image_tokens = ie_config.num_mm_tokens_per_image  # 256
    image_size = ie_config.image_embedding.image_size  # 896

    # 더미 토큰: [BOS] + 256개 이미지 토큰 + 텍스트 토큰
    text_tokens = [2, 651, 9456, 576, 573]
    image_token_id = config_mm.image_token_id  # 262144
    token_ids = text_tokens[:1] + [image_token_id] * num_image_tokens + text_tokens[1:]
    seq_len = len(token_ids)

    tokens = torch.tensor([token_ids], dtype=torch.int)
    input_pos = torch.arange(0, seq_len, dtype=torch.int)

    # 더미 이미지 (1장)
    pixel_values = torch.randn(1, 1, 3, image_size, image_size)

    # 이미지 인덱스: 이미지 토큰 위치에 0 (첫 번째 이미지), 나머지는 -1
    image_indices = torch.full((1, seq_len), -1, dtype=torch.long)
    image_indices[0, 1:1+num_image_tokens] = 0  # 첫 번째 이미지

    # 이미지 피처 인덱스: 각 이미지 토큰이 어떤 패치에 대응하는지
    image_feat_indices = torch.full((1, seq_len), 0, dtype=torch.long)
    image_feat_indices[0, 1:1+num_image_tokens] = torch.arange(num_image_tokens)

    kv_cache = kv_utils.KVCache.from_model_config(
        kv_cache_max_len, dec_config,
        kv_layout=kv_utils.KV_LAYOUT_TRANSPOSED,
    )

    output = model(
        tokens=tokens,
        input_pos=input_pos,
        kv_cache=kv_cache,
        image_indices=image_indices,
        image_feat_indices=image_feat_indices,
        pixel_values=pixel_values,
    )

    logits = output["logits"]
    print(f"MM vision+text logits shape: {logits.shape}")
    print(f"Expected: (1, {seq_len}, {dec_config.vocab_size})")
    assert logits.shape == (1, seq_len, dec_config.vocab_size), "Shape mismatch!"
    print("\nMM vision+text forward pass: PASSED")
    return output


if os.path.exists(CHECKPOINT_PATH):
    verify_mm_with_image(gemma3_4b_mm_model, config_4b_mm, KV_CACHE_MAX_LEN)
else:
    print("Skipping - checkpoint not available.")

## 6. TFLite 변환 (텍스트 전용)

4B Decoder-only 모델을 multi-signature TFLite로 변환합니다.

### 시그니처 구성
- `prefill_{seq_len}`: 입력 토큰 시퀀스 전체를 한 번에 처리
- `decode`: 토큰 1개씩 자기회귀적으로 생성

### Gemma 3 특수 설정
- `mask_as_input=True`: Global+Local 어텐션 패턴 지원
- `transpose_kv_cache=True`: KV 캐시 전치 형식 사용

In [ ]:
@torch.inference_mode()
def convert_gemma3_4b_to_tflite(
    model: nn.Module,
    output_dir: str,
    prefill_seq_lens: list = None,
    kv_cache_max_len: int = 1280,
    quantize: str = "dynamic_int8",
    mask_as_input: bool = True,
    transpose_kv_cache: bool = True,
) -> str:
    """Gemma 3 4B decoder 모델을 multi-signature TFLite로 변환합니다."""
    if prefill_seq_lens is None:
        prefill_seq_lens = [8, 64, 128, 256, 512, 1024]

    config = decoder.get_decoder_config_4b()

    kv_layout = (
        kv_utils.KV_LAYOUT_TRANSPOSED
        if transpose_kv_cache
        else kv_utils.KV_LAYOUT_DEFAULT
    )
    export_config = export_cfg.ExportConfig(
        mask_as_input=mask_as_input,
        kvcache_layout=kv_layout,
    )

    output_file = conv_utils.convert_to_tflite(
        pytorch_model=model,
        output_path=output_dir,
        output_name_prefix="gemma3-4b",
        prefill_seq_len=prefill_seq_lens,
        kv_cache_max_len=kv_cache_max_len,
        quantize=quantize,
        config=config,
        export_config=export_config,
    )

    print(f"TFLite model saved to: {output_file}")
    size_mb = os.path.getsize(output_file) / (1024 * 1024)
    print(f"File size: {size_mb:.1f} MB")
    return output_file


print("Conversion function defined.")

In [ ]:
# 텍스트 전용 TFLite 변환 실행
if os.path.exists(CHECKPOINT_PATH):
    print("Starting 4B text-only conversion with dynamic INT8 quantization...")
    print(f"  Prefill lengths: {PREFILL_SEQ_LENS}")
    print(f"  KV cache max: {KV_CACHE_MAX_LEN}")
    print()

    tflite_path_4b = convert_gemma3_4b_to_tflite(
        model=gemma3_4b_model,
        output_dir=OUTPUT_DIR,
        prefill_seq_lens=PREFILL_SEQ_LENS,
        kv_cache_max_len=KV_CACHE_MAX_LEN,
        quantize="dynamic_int8",
    )
else:
    print("Skipping conversion - checkpoint not available.")

## 7. TFLite 변환 (멀티모달)

4B MM 모델의 디코더 부분만 TFLite로 변환합니다.
Vision Encoder는 별도로 변환하거나, 디코더와 함께 전체 모델로 변환할 수 있습니다.

In [ ]:
# MM 모델의 디코더 부분만 TFLite 변환
if os.path.exists(CHECKPOINT_PATH):
    print("Starting 4B MM decoder conversion with dynamic INT8 quantization...")
    print(f"  Prefill lengths: {PREFILL_SEQ_LENS}")
    print(f"  KV cache max: {KV_CACHE_MAX_LEN}")
    print()

    # MM 모델의 decoder 부분만 추출하여 변환
    tflite_path_4b_mm = convert_gemma3_4b_to_tflite(
        model=gemma3_4b_mm_model.decoder,
        output_dir=OUTPUT_DIR,
        prefill_seq_lens=PREFILL_SEQ_LENS,
        kv_cache_max_len=KV_CACHE_MAX_LEN,
        quantize="dynamic_int8",
    )
else:
    print("Skipping conversion - checkpoint not available.")

## 8. TFLite 모델로 텍스트 생성

변환된 TFLite 모델을 사용한 자기회귀 텍스트 생성 파이프라인입니다.

### 추론 흐름

```
1. Tokenize(prompt) → token_ids
2. Prefill: model(token_ids, input_pos, kv_cache, mask) → logits, kv_cache'
3. Sample: argmax(logits[-1]) → next_token
4. Decode loop:
   model(next_token, pos, kv_cache', mask) → logits, kv_cache''
   argmax(logits) → next_token
   반복...
5. Detokenize → 텍스트
```

In [ ]:
# PyTorch 모델로 직접 텍스트 생성 (TFLite 변환 전 검증용)

@torch.inference_mode()
def generate_with_pytorch(
    model: nn.Module,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 30,
    kv_cache_max_len: int = 1280,
) -> str:
    """PyTorch 모델로 텍스트를 생성합니다."""
    config = model.config

    chat_prompt = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    input_ids = [2] + tokenizer.encode(chat_prompt)
    tokens = torch.tensor([input_ids], dtype=torch.int)
    seq_len = tokens.shape[1]

    kv_cache = kv_utils.KVCache.from_model_config(
        kv_cache_max_len, config,
        kv_layout=kv_utils.KV_LAYOUT_TRANSPOSED,
    )

    mask_cache = attn_utils.build_causal_mask_cache(kv_cache_max_len)

    # Prefill
    input_pos = torch.arange(0, seq_len, dtype=torch.int)
    mask = mask_cache.index_select(2, input_pos)
    output = model(tokens, input_pos, kv_cache, mask=mask)
    logits = output["logits"]
    kv_cache = output["kv_cache"]

    next_token = logits[0, -1].argmax().item()
    input_ids.append(next_token)

    # Decode 루프
    for i in range(max_new_tokens - 1):
        tokens = torch.tensor([[next_token]], dtype=torch.int)
        input_pos = torch.tensor([len(input_ids) - 1], dtype=torch.int)
        mask = mask_cache.index_select(2, input_pos)
        output = model(tokens, input_pos, kv_cache, mask=mask)
        logits = output["logits"]
        kv_cache = output["kv_cache"]

        next_token = logits[0, -1].argmax().item()
        input_ids.append(next_token)

        if next_token == 1:  # <eos>
            break

    return tokenizer.decode(input_ids)


print("Generation function defined.")

In [ ]:
# SentencePiece 토크나이저를 사용한 생성 예시

try:
    import sentencepiece as spm

    tokenizer_path = os.path.join(CHECKPOINT_PATH, "tokenizer.model")
    if os.path.exists(tokenizer_path) and os.path.exists(CHECKPOINT_PATH):
        sp = spm.SentencePieceProcessor()
        sp.Load(tokenizer_path)
        print(f"Tokenizer loaded: vocab_size={sp.GetPieceSize()}")

        prompt = "What is the meaning of life?"
        print(f"\nPrompt: {prompt}")
        result = generate_with_pytorch(
            gemma3_4b_model, sp, prompt,
            max_new_tokens=50,
            kv_cache_max_len=KV_CACHE_MAX_LEN,
        )
        print(f"Output: {result}")
    else:
        print(f"Tokenizer not found at: {tokenizer_path}")
except ImportError:
    print("sentencepiece not installed. Run: pip install sentencepiece")

In [ ]:
# TFLite 모델로 텍스트 생성

import numpy as np


def generate_with_tflite(
    tflite_path: str,
    tokenizer_path: str,
    prompt: str,
    max_new_tokens: int = 50,
    kv_cache_max_len: int = 1280,
) -> str:
    """TFLite 모델로 텍스트를 생성합니다."""
    from ai_edge_litert import interpreter as tflite_interp
    import sentencepiece as spm

    sp = spm.SentencePieceProcessor()
    sp.Load(tokenizer_path)

    interp = tflite_interp.Interpreter(model_path=tflite_path)
    interp.allocate_tensors()
    sig_list = interp.get_signature_list()
    print(f"Available signatures: {list(sig_list.keys())}")

    # 토크나이즈
    chat_prompt = f"<start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    token_ids = [2] + sp.encode(chat_prompt)
    print(f"Prompt tokens ({len(token_ids)}): {token_ids[:20]}...")

    # Prefill 시그니처 선택
    prefill_sigs = sorted(
        [k for k in sig_list if k.startswith("prefill_")],
        key=lambda k: int(k.split("_")[1]),
    )
    prefill_sig_name = None
    prefill_seq_len = None
    for sig_name in prefill_sigs:
        seq_len = int(sig_name.split("_")[1])
        if seq_len >= len(token_ids) - 1:
            prefill_sig_name = sig_name
            prefill_seq_len = seq_len
            break
    if prefill_sig_name is None:
        prefill_sig_name = prefill_sigs[-1]
        prefill_seq_len = int(prefill_sig_name.split("_")[1])
    print(f"Using signature: {prefill_sig_name} (seq_len={prefill_seq_len})")

    # KV 캐시 초기화
    decode_runner = interp.get_signature_runner("decode")
    decode_input_details = sig_list["decode"]["inputs"]
    kv_cache = {}
    for input_name in decode_input_details:
        if input_name.startswith("kv_cache_"):
            input_detail = decode_runner.get_input_details()[input_name]
            kv_cache[input_name] = np.zeros(input_detail["shape"], dtype=input_detail["dtype"])

    print(f"KV cache entries: {len(kv_cache)} (layers: {len(kv_cache) // 2})")
    if kv_cache:
        sample_key = list(kv_cache.keys())[0]
        print(f"  {sample_key} shape: {kv_cache[sample_key].shape}")

    # Causal mask 빌드
    def build_causal_mask(seq_len, kv_max_len, start_pos=0):
        mask = np.full((seq_len, kv_max_len), float("-inf"), dtype=np.float32)
        for i in range(seq_len):
            mask[i, : start_pos + i + 1] = 0.0
        return mask.reshape(1, 1, seq_len, kv_max_len)

    # Prefill 실행
    effective_len = min(len(token_ids), prefill_seq_len + 1)
    prefill_len = effective_len - 1

    prefill_tokens = np.zeros((1, prefill_seq_len), dtype=np.int32)
    prefill_input_pos = np.zeros(prefill_seq_len, dtype=np.int32)
    for i in range(prefill_len):
        prefill_tokens[0, i] = token_ids[i]
        prefill_input_pos[i] = i

    prefill_kwargs = {
        "tokens": prefill_tokens,
        "input_pos": prefill_input_pos,
        **kv_cache,
    }
    prefill_input_details = sig_list[prefill_sig_name]["inputs"]
    if "mask" in prefill_input_details:
        prefill_kwargs["mask"] = build_causal_mask(prefill_seq_len, kv_cache_max_len)

    prefill_runner = interp.get_signature_runner(prefill_sig_name)
    prefill_output = prefill_runner(**prefill_kwargs)

    for key in kv_cache:
        if key in prefill_output:
            kv_cache[key] = prefill_output[key]

    # Decode 루프
    next_token = token_ids[prefill_len]
    next_pos = prefill_len
    output_tokens = []
    eos_tokens = {1, sp.piece_to_id("<end_of_turn>")}

    for step in range(max_new_tokens):
        decode_tokens = np.array([[next_token]], dtype=np.int32)
        decode_input_pos = np.array([next_pos], dtype=np.int32)

        decode_kwargs = {
            "tokens": decode_tokens,
            "input_pos": decode_input_pos,
            **kv_cache,
        }
        if "mask" in sig_list["decode"]["inputs"]:
            decode_kwargs["mask"] = build_causal_mask(1, kv_cache_max_len, start_pos=next_pos)

        decode_runner = interp.get_signature_runner("decode")
        decode_output = decode_runner(**decode_kwargs)

        for key in kv_cache:
            if key in decode_output:
                kv_cache[key] = decode_output[key]

        logits = decode_output["logits"]
        next_token = int(np.argmax(logits[0, -1, :]))
        output_tokens.append(next_token)
        next_pos += 1

        if next_token in eos_tokens:
            break

    output_text = sp.decode(output_tokens)
    return output_text


print("TFLite generation function defined.")

In [ ]:
# TFLite 모델로 추론 실행
if os.path.exists(CHECKPOINT_PATH):
    tokenizer_path = os.path.join(CHECKPOINT_PATH, "tokenizer.model")

    prompt = "What is the meaning of life?"
    print(f"Prompt: {prompt}\n")

    result = generate_with_tflite(
        tflite_path=tflite_path_4b,
        tokenizer_path=tokenizer_path,
        prompt=prompt,
        max_new_tokens=100,
        kv_cache_max_len=KV_CACHE_MAX_LEN,
    )
    print(f"\nGenerated: {result}")
else:
    print("Skipping - checkpoint not available.")

## 9. CLI 변환 스크립트 사용법

노트북 대신 CLI 스크립트로도 변환할 수 있습니다.

### 4B 텍스트 전용
```bash
python litert_torch/generative/examples/gemma3/convert_gemma3_to_tflite.py \
    --model_size=4b \
    --checkpoint_path=~/gemma-3-4b-it/ \
    --output_path=/tmp/gemma3_4b_tflite/ \
    --output_name_prefix=gemma3-4b \
    --quantize=dynamic_int8 \
    --prefill_seq_lens=8,64,128,256,512,1024 \
    --kv_cache_max_len=1280 \
    --mask_as_input=true \
    --transpose_kv_cache=true
```

### 4B 멀티모달
```bash
python litert_torch/generative/examples/gemma3/convert_gemma3_to_tflite.py \
    --model_size=4b_mm \
    --checkpoint_path=~/gemma-3-4b-it/ \
    --output_path=/tmp/gemma3_4b_mm_tflite/ \
    --output_name_prefix=gemma3-4b-mm \
    --quantize=dynamic_int8
```

### INT4 양자화
```bash
python litert_torch/generative/examples/gemma3/convert_gemma3_to_tflite.py \
    --model_size=4b \
    --checkpoint_path=~/gemma-3-4b-it/ \
    --quantize=dynamic_int4_block32
```

## 요약

이 노트북에서 다뢬 내용:

1. Gemma 3 4B **아키텍처 분석** (34 layers, GQA 8/4, sliding window 1024)
2. **HuggingFace 4B 테서 이름 매핑** (`language_model.model.xxx`, `vision_tower.xxx`)
3. **4B 텍스트 전용 모델** 빌드 및 forward pass 검증
4. **4B 멀티모달 모델** 빌드 및 검증 (Vision Encoder + Projection + Decoder)
5. **Projection 전치** 처리 (`[1152, 2560]` → `[2560, 1152]`)
6. **multi-signature TFLite 변환** (prefill + decode)
7. KV 캐시 기반 **자기회귀 텍스트 생성** 파이프라인 (PyTorch + TFLite)
8. **CLI 스크립트** 사용법 (`--model_size=4b` / `--model_size=4b_mm`)